# GNN MAML Model Evaluation V2

이 노트북은 pretrained된 GNN MAML 모델을 test data로 평가합니다.
- 참조 노트북과 동일한 center point shift + SGD/Adam finetuning 방식
- Train/Test split된 데이터 사용
- 연속성 체크 포함
- Global R² score 및 NRMSE 계산

In [9]:
import os
import torch
from torch import optim
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
import matplotlib.pyplot as plt
import time
import math
import csv
from collections import OrderedDict
from copy import deepcopy

In [10]:
# 연속성 체크 함수들
def check_strict_continuity(data, threshold_ratio=0.2, min_points=5):
    """
    매우 엄격한 연속성을 체크합니다. 이웃한 점들 사이에서 하나라도 임계값을 넘는 점프가 있으면 불연속으로 판단합니다.
    """
    if len(data.shape) > 1:
        data = data.flatten()
    
    data = np.array(data)
    
    # NaN이나 inf 값 제거
    valid_mask = np.isfinite(data)
    if not np.any(valid_mask):
        return False, 0.0, [], 0.0, 0.0
    
    valid_data = data[valid_mask]
    
    if len(valid_data) < min_points:
        return False, 0.0, [], 0.0, 0.0
    
    # 데이터 범위 계산
    data_range = np.max(valid_data) - np.min(valid_data)
    if data_range == 0:
        return True, 1.0, [], 0.0, 0.0  # 상수 데이터는 연속으로 간주
    
    # 차분 계산 (이웃한 점들 간의 차이)
    diff = np.diff(valid_data)
    threshold = data_range * threshold_ratio
    
    # 절댓값으로 큰 점프 찾기
    abs_diff = np.abs(diff)
    max_jump = np.max(abs_diff)
    max_jump_ratio = max_jump / threshold if threshold > 0 else 0
    
    # 큰 점프 찾기
    large_jumps = abs_diff > threshold
    gap_indices = np.where(large_jumps)[0]
    
    # 엄격한 연속성 판단: 하나라도 큰 점프가 있으면 불연속
    is_continuous = len(gap_indices) == 0
    
    # 연속성 점수 계산
    continuity_score = 1.0 - (len(gap_indices) / len(diff)) if len(diff) > 0 else 1.0
    
    return is_continuous, continuity_score, gap_indices, max_jump, max_jump_ratio

print("✅ 연속성 체크 함수 정의 완료")

✅ 연속성 체크 함수 정의 완료


In [11]:
# GPU 설정
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)
print('Current cuda device:', torch.cuda.current_device())
print('Count of using GPUs:', torch.cuda.device_count())

Device: cuda
Current cuda device: 0
Count of using GPUs: 1


In [12]:
# Import GNN MAML modules
import sys
sys.path.append('./')
sys.path.append('tools/data_processing')

from gnn_maml_optimized_v2 import (
    MAML_GNN_Model,
    create_maml_gcn_model
)
from maml_gnn_training_optimized_fixed import (
    GNNOptimizedMAML,
    calculate_norm_stats_from_data,
    load_gnn_data_for_maml
)

Device: cuda
Current cuda device: 0
Count of using GPUs: 1
GPU name: NVIDIA GeForce RTX 3090


In [13]:
def plot_sampled_performance_gnn(initial_model, support_graphs, support_outputs, all_task_graphs, all_task_outputs,
                                 grad, move, mean ,std ,norm_stats=None, lr=0.01, adam_step=50):
    """
    참조 노트북의 plot_sampled_performance와 동일한 구조로 GNN 모델 평가
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    def normalize_node_features(node_features):
        """Normalize node features using saved statistics"""
        if norm_stats is None:
            return node_features
            
        normalized = node_features.clone()
        
        # Normalize voltage (column 4)
        voltage_mask = normalized[:, 4] != 0
        if voltage_mask.any():
            normalized[voltage_mask, 4] = (
                normalized[voltage_mask, 4] - norm_stats['node_features']['voltage']['mean']
            ) / norm_stats['node_features']['voltage']['std']
        
        # Normalize input_slew (column 5)
        slew_mask = normalized[:, 5] != 0
        if slew_mask.any():
            normalized[slew_mask, 5] = (
                normalized[slew_mask, 5] - norm_stats['node_features']['input_slew']['mean']
            ) / norm_stats['node_features']['input_slew']['std']
        
        # Normalize output_load (column 6)
        load_mask = normalized[:, 6] != 0
        if load_mask.any():
            normalized[load_mask, 6] = (
                normalized[load_mask, 6] - norm_stats['node_features']['output_load']['mean']
            ) / norm_stats['node_features']['output_load']['std']
        
        return normalized
    
    def create_pyg_data(graph_sample):
        """Create PyTorch Geometric Data object"""
        node_features = graph_sample['node_features']
        adjacency_matrix = graph_sample['adjacency_matrix']
        edge_index = graph_sample['edge_index']
        
        # Apply normalization
        normalized_features = normalize_node_features(node_features)
        # Apply adjacency matrix multiplication (A × X)
        aggregated_features = torch.matmul(adjacency_matrix, normalized_features)
        
        data = Data(
            x=aggregated_features,
            edge_index=edge_index
        )
        
        return data
    
    # Copy model to preserve MAML weights
    model = deepcopy(initial_model)
    model.to(device)
    criterion = nn.MSELoss()
    
    # Task statistics
    y_mean = torch.tensor(all_task_outputs).mean().item()
    y_std = torch.tensor(all_task_outputs).std().item()
    if y_std < 1e-8:
        y_std = 1.0
    #print(support_outputs)
    # Prepare support data with scaling
    support_outputs_scaled = torch.tensor(support_outputs, dtype=torch.float32).to(device).view(-1, 1)

    support_outputs_scaled = (support_outputs_scaled) / grad + move
    #print(support_outputs_scaled)    
    # SGD training (10 steps)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=1e-4)
    losses = []
    K = len(support_graphs)
    
    # for step in range(1, 31):  # 10 SGD steps
    #     batch_data = []
    #     for graph in support_graphs:
    #         data = create_pyg_data(graph)
    #         batch_data.append(data)

    #     X = Batch.from_data_list(batch_data).to(device)
    #     if(step==1):
    #         print(f"model_prefixed:{model(X)}")
    #     # print(f"model:{model(X)}")
    #     # print(support_outputs_scaled)
    #     loss = criterion(model(X), support_outputs_scaled) / K
    #     print(loss)
    #     losses.append(loss.item())
    #     model.zero_grad()
    #     loss.backward()
    #     optimizer.step()
    
    # Adam training if SGD loss is still high
    #if losses[-1] > 1e-5 and adam_step > 0:

    optimizer2 = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=5e-5)
    batch_data = []
    for graph in support_graphs:
        data = create_pyg_data(graph)
        batch_data.append(data)
        X = Batch.from_data_list(batch_data).to(device)
    loss = criterion(model(X), support_outputs_scaled) / K
    #print(loss)
    if(loss>1e-3):
        #print("hi")
        for step in range(1, adam_step + 1):  # 30 Adam steps
            
            # if(step==1):
            #     print(f"model_prefixed:{model(X)}")
            loss = criterion(model(X), support_outputs_scaled) / K
            #print(loss)
            #print(loss)
            losses.append(loss.item())
            model.zero_grad()
            loss.backward()
            optimizer2.step()
    # print(f"model:{model(X)}")
    # print(support_outputs_scaled)
    # Evaluate on all task data (like 61 points in reference)
    total_loss = 0
    total_inter_loss = 0
    total_rightex_loss = 0
    total_leftex_loss = 0
    total_mape_loss = 0
    total_leftex_mape = 0
    total_inter_mape = 0
    total_rightex_mape = 0
    
    predictions = []
    actual_values = []
    
    num_points = len(all_task_graphs)
    leftex_boundary = int(num_points * 0.164)  # ~10/61
    rightex_boundary = int(num_points * 0.82)   # ~50/61
    
    for i, (graph, actual_output) in enumerate(zip(all_task_graphs, all_task_outputs)):
        # Get model prediction
        data = create_pyg_data(graph)
        X_single = Batch.from_data_list([data]).to(device)
        #print(graph)
        with torch.no_grad():
            pred_scaled = model(X_single).item()
        
        # Denormalize prediction
        pred_value = (pred_scaled - move) * std * grad  + mean
        actual_value = actual_output
        # print(f"pred:{i}:{pred_value}")
        # print(f"act:{actual_value}")
        predictions.append(pred_value)
        actual_values.append(actual_value)
        
        # Calculate loss
        loss = criterion(
            torch.tensor([pred_value], dtype=torch.float32),
            torch.tensor([actual_value], dtype=torch.float32)
        )
        
        # Calculate MAPE
        if abs(actual_value) > 1e-8:
            mape_loss = abs((pred_value - actual_value) / actual_value)
        else:
            mape_loss = 0
        
        total_loss += loss
        total_mape_loss += mape_loss
        
        # Regional calculations (like reference notebook)
        if i < leftex_boundary:  # Left extrapolation
            total_leftex_loss += loss
            total_leftex_mape += mape_loss
        elif i < rightex_boundary:  # Interpolation
            total_inter_loss += loss
            total_inter_mape += mape_loss
        else:  # Right extrapolation
            total_rightex_loss += loss
            total_rightex_mape += mape_loss
    # Calculate average losses
    avg_total_loss = total_loss / num_points
    avg_inter_loss = total_inter_loss / (rightex_boundary - leftex_boundary) if rightex_boundary > leftex_boundary else 0
    avg_rightex_loss = total_rightex_loss / (num_points - rightex_boundary) if num_points > rightex_boundary else 0
    avg_leftex_loss = total_leftex_loss / leftex_boundary if leftex_boundary > 0 else 0
    avg_total_mape = total_mape_loss / num_points
    avg_leftex_mape = total_leftex_mape / leftex_boundary if leftex_boundary > 0 else 0
    avg_inter_mape = total_inter_mape / (rightex_boundary - leftex_boundary) if rightex_boundary > leftex_boundary else 0
    avg_rightex_mape = total_rightex_mape / (num_points - rightex_boundary) if num_points > rightex_boundary else 0
    
    return (avg_total_loss, avg_inter_loss, avg_leftex_loss, avg_rightex_loss,
            avg_total_mape, avg_leftex_mape, avg_inter_mape, avg_rightex_mape,
            predictions, actual_values, model, y_mean, y_std, losses)

print("✅ GNN evaluation function 정의 완료 (참조 노트북과 동일한 구조)")

✅ GNN evaluation function 정의 완료 (참조 노트북과 동일한 구조)


In [14]:
# Load test data from train_test_split
print("📊 Loading test data...")
process_type = "SLVT"  # 또는 "RVT", "SLVT", "SRAM"
corner_type = "TT"    # 또는 "FF", "TT"

base_path = "dataset_gnn/processed_batch"
matching_folders = []

# Find matching test folders
for item in os.listdir(base_path):
    parts = item.split('_')
    dataset_process = None
    for part in parts:
        if part in ['RVT', 'LVT', 'SLVT', 'SRAM']:
            dataset_process = part
            break
    
    if dataset_process != process_type:
        continue
    
    # Check corner type matching
    corner_match = False
    if corner_type == "TT" and not item.endswith(('_FF', '_SS')):
        corner_match = True
    elif corner_type == "FF" and item.endswith('_FF'):
        corner_match = True
    elif corner_type == "SS" and item.endswith('_SS'):
        corner_match = True
    
    if corner_match:
        test_data_path = f"{base_path}/{item}/train_test_split/test_data.pth"
        if os.path.exists(test_data_path):
            matching_folders.append((item, test_data_path))

print(f"   📂 Found {len(matching_folders)} matching test datasets:")
for folder, _ in matching_folders:
    print(f"     - {folder}")

# Load and merge test data
all_graph_data_per_file = []
all_stacked_outputs = []

for folder, test_data_path in matching_folders:
    print(f"   📥 Loading: {folder}")
    
    data = torch.load(test_data_path, weights_only=False, map_location='cpu')
    graph_data_per_file = data['graph_data_per_file']
    stacked_outputs = data['stacked_outputs']
    
    print(f"     Tasks: {stacked_outputs.shape[0]}, Lib files: {stacked_outputs.shape[1]}")
    
    if not all_graph_data_per_file:
        # First dataset - initialize
        all_graph_data_per_file = [[] for _ in range(len(graph_data_per_file))]
        
    # Merge graph data
    for lib_idx, lib_graphs in enumerate(graph_data_per_file):
        all_graph_data_per_file[lib_idx].extend(lib_graphs)
    
    # Collect stacked outputs
    all_stacked_outputs.append(stacked_outputs)

# Concatenate all outputs
test_stacked_outputs = torch.cat(all_stacked_outputs, dim=0)

print(f"   ✅ Merged test data:")
print(f"     Total tasks: {test_stacked_outputs.shape[0]} (input conditions)")
print(f"     Lib files per task: {test_stacked_outputs.shape[1]} (process variations)")

# Calculate norm_stats from test data
norm_stats = calculate_norm_stats_from_data(all_graph_data_per_file)

📊 Loading test data...
   📂 Found 2 matching test datasets:
     - INVBUF_SLVT
     - simple_SLVT
   📥 Loading: INVBUF_SLVT


     Tasks: 647, Lib files: 61
   📥 Loading: simple_SLVT
     Tasks: 3058, Lib files: 61
   ✅ Merged test data:
     Total tasks: 3705 (input conditions)
     Lib files per task: 61 (process variations)
   🔍 Calculating norm_stats from 61 lib files...
   📊 Sampled 700 graphs
     Voltage: mean=0.700000, std=0.200000 (n=7070)
     Input Slew: mean=88.633663, std=107.312417 (n=7070)
     Output Load: mean=86.309601, std=151.052700 (n=1400)
     Delay: mean=0.268165, std=0.326045 (n=2163)


In [15]:
# 연속성 체크
print("🔍 테스트 데이터 연속성 체크...")

threshold_ratio = 0.18
continuous_task_ids = []
discontinuous_task_ids = []

# 샘플링: 처음 1000개만 체크 (빠른 테스트)
num_check_samples = min(1000, len(test_stacked_outputs))
check_indices = list(range(num_check_samples))

print(f"처음 {num_check_samples}개 태스크에 대해 연속성 분석...")

for i, task_id in enumerate(check_indices):
    if i % 100 == 0:
        print(f"진행 상황: {i+1}/{num_check_samples}")
    
    try:
        # 출력 데이터 연속성 체크
        output_data = test_stacked_outputs[task_id].cpu().numpy()
        output_continuous, output_score, output_gaps, output_max_jump, output_max_ratio = check_strict_continuity(
            output_data, threshold_ratio=threshold_ratio
        )
        
        if output_continuous:
            continuous_task_ids.append(task_id)
        else:
            discontinuous_task_ids.append(task_id)
            
    except Exception as e:
        print(f"Error processing task {task_id}: {e}")
        continue

print(f"\n📊 연속성 분석 완료!")
print(f"   • 연속적인 태스크: {len(continuous_task_ids)} ({len(continuous_task_ids)/num_check_samples*100:.1f}%)")
print(f"   • 불연속적인 태스크: {len(discontinuous_task_ids)} ({len(discontinuous_task_ids)/num_check_samples*100:.1f}%)")

if len(continuous_task_ids) > 0:
    print(f"   • 처음 10개 연속적인 태스크: {continuous_task_ids[:10]}")

🔍 테스트 데이터 연속성 체크...
처음 1000개 태스크에 대해 연속성 분석...
진행 상황: 1/1000
진행 상황: 101/1000
진행 상황: 201/1000
진행 상황: 301/1000
진행 상황: 401/1000
진행 상황: 501/1000
진행 상황: 601/1000
진행 상황: 701/1000
진행 상황: 801/1000
진행 상황: 901/1000

📊 연속성 분석 완료!
   • 연속적인 태스크: 1000 (100.0%)
   • 불연속적인 태스크: 0 (0.0%)
   • 처음 10개 연속적인 태스크: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [19]:
# Load pretrained GNN MAML model
print("🤖 Loading pretrained GNN MAML model...")

# Model configuration (should match training)
layer_length = 128
inner_steps = 1
K = 5  # Support set size

# Load model
gnn_model = create_maml_gcn_model(
    node_features=7,
    hidden_dim=layer_length,
    num_layers=3,
    pooling='mean',
    output_dim=1
).to(device)

# gnn_model = create_maml_gcn_bn_model(
#     node_features=7,
#     hidden_dim=layer_length,
#     num_layers=3,
#     pooling='mean',
#     output_dim=1
# ).to(device)
# Load trained weights
checkpoint_path = f"pretrained_models/gnn_maml_checkpoints/gnn_maml_SLVT_{corner_type}_chunk_6_K1.pth"
# 또는 최종 모델
# checkpoint_path = f"pretrained_models/gnn_maml_final/gnn_maml_{process_type}_{corner_type}_final_K5.pth"

try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    if 'model_state_dict' in checkpoint:
        gnn_model.load_state_dict(checkpoint['model_state_dict'])
        config = checkpoint.get('config', {})
        saved_norm_stats = checkpoint.get('norm_stats', norm_stats)
        print(f"   ✅ Loaded from checkpoint: {checkpoint_path}")
        print(f"   📊 Model config: {config}")
    else:
        gnn_model.load_state_dict(checkpoint)
        saved_norm_stats = norm_stats
        print(f"   ✅ Loaded state dict: {checkpoint_path}")
        
except FileNotFoundError:
    print(f"   ⚠️ Checkpoint not found: {checkpoint_path}")
    print(f"   🔄 Using randomly initialized model for demonstration")
    saved_norm_stats = norm_stats
except Exception as e:
    print(f"   ❌ Error loading checkpoint: {e}")
    print(f"   🔄 Using randomly initialized model for demonstration")
    saved_norm_stats = norm_stats

gnn_model.eval()
print(f"   🎯 Model ready for evaluation")

🤖 Loading pretrained GNN MAML model...
   ✅ Loaded from checkpoint: pretrained_models/gnn_maml_checkpoints/gnn_maml_SLVT_TT_chunk_6_K1.pth
   📊 Model config: {'process_type': 'SLVT', 'corner_type': 'TT', 'layer_length': 128, 'inner_steps': 1, 'K': 5, 'iterations_completed': 60000, 'meta_losses': [0.19310054332017898, 0.18073811531066894, 0.18772799223661424, 0.1778217226266861, 0.20048796236515046, 0.1723321110010147, 0.16110416501760483, 0.16216477751731873, 0.16498489901423455, 0.1451333574950695, 0.15299486592411995, 0.11752573698759079, 0.12582307904958726, 0.14407441914081573, 0.12924388721585273, 0.13915128111839295, 0.11428850218653679, 0.12448246106505394, 0.11986861042678357, 0.11479740962386131, 0.12965481914579868, 0.12278558015823364, 0.10021127685904503, 0.10194299519062042, 0.10436200127005577, 0.11426932625472545, 0.1051059290766716, 0.09605691321194172, 0.10147047489881515, 0.10302868634462356, 0.0870207816362381, 0.076444810628891, 0.08923989273607731, 0.10754264332354

In [17]:
# Select K support points (like reference notebook)
K = 5
indices = [0, 13, 30, 45, 60]  # Same indices as reference notebook

# Evaluation on continuous tasks
print(f"🔄 Evaluating on {len(continuous_task_ids)} continuous tasks...")

# Use subset for faster evaluation
#num_test_samples = min(100, len(continuous_task_ids))  # 처음 100개로 제한
num_test_samples = min(100, len(continuous_task_ids))
test_indices = random.sample(range(len(continuous_task_ids)), num_test_samples)

print(f"🎲 Selected {num_test_samples} continuous test tasks")

# Global collections for R² analysis
total_nrmse = []
total_extra_l = []
total_extra_r = []
total_inter = []
total_mape = []
total_mape_l = []  # Left extrapolation MAPE
total_mape_r = []  # Right extrapolation MAPE
total_mape_inter = []  # Interpolation MAPE
high_error = []

all_predictions_global = []
all_actuals_global = []

testdata_output = test_stacked_outputs.to(device)
testdata_input = all_graph_data_per_file  # Graph data

🔄 Evaluating on 1000 continuous tasks...
🎲 Selected 100 continuous test tasks


In [20]:
# Process tasks (similar to reference notebook)
print(f"🔄 Processing {num_test_samples} continuous tasks...")

for i, task_id in enumerate(test_indices):
    if i % 10 == 0:
        print(f"Processing task {i+1}/{num_test_samples} (index: {task_id})")
    
    try:
        # Get support and all task data
        support_graphs = []
        support_outputs = []
        all_task_graphs = []
        all_task_outputs = []
        
        # Collect data from all lib files for this task
        for lib_idx in range(len(testdata_input)):
            if task_id < len(testdata_input[lib_idx]):
                graph = testdata_input[lib_idx][task_id]
                output = testdata_output[task_id, lib_idx].item()
                
                all_task_graphs.append(graph)
                all_task_outputs.append(output)
                
                # Select support set using same indices as reference
                if lib_idx in indices:
                    support_graphs.append(graph)
                    support_outputs.append(output)
        
        if len(support_graphs) < K or len(all_task_graphs) < 61:
            continue
        #print(support_graphs)
        # Define regions (matching reference notebook)
        testdata_rightex_output = [all_task_outputs[i] for i in range(50, len(all_task_outputs))]
        testdata_leftex_output = [all_task_outputs[i] for i in range(10)]
        testdata_inter_output = [all_task_outputs[i] for i in range(10, 50)]
        
        y_leftex_mean = np.mean(testdata_leftex_output) if testdata_leftex_output else 0
        y_rightex_mean = np.mean(testdata_rightex_output) if testdata_rightex_output else 0
        y_inter_mean = np.mean(testdata_inter_output) if testdata_inter_output else 0
        y1_mean = np.mean(all_task_outputs)
        
        # Normalize support outputs
        y_mean = np.mean(support_outputs)
        y_std = np.std(support_outputs)
        if y_std > 1e-8:  # Only process if there's variation
            y_norm = [(y - y_mean) / y_std for y in support_outputs]
            
            # Create center input (voltage = 0)
            def create_center_graph(base_graph):
                center_graph = deepcopy(base_graph)
                center_features = center_graph['node_features'].clone()
                center_features[:, 4] = 0.0  # Set voltage to 0
                center_graph['node_features'] = center_features
                return center_graph
            
            def create_pyg_data_simple(graph_sample, norm_stats):
                """Simple PyG data creation for center point"""
                node_features = graph_sample['node_features']
                adjacency_matrix = graph_sample['adjacency_matrix']
                edge_index = graph_sample['edge_index']
                
                # Normalize features
                normalized = node_features.clone()
                if norm_stats is not None:
                    # Normalize voltage (column 4)
                    voltage_mask = normalized[:, 4] != 0
                    if voltage_mask.any() and 'voltage' in norm_stats['node_features']:
                        normalized[voltage_mask, 4] = (
                            normalized[voltage_mask, 4] - norm_stats['node_features']['voltage']['mean']
                        ) / norm_stats['node_features']['voltage']['std']
                    slew_mask = normalized[:, 5] != 0
                    if slew_mask.any():
                        normalized[slew_mask, 5] = (
                            normalized[slew_mask, 5] - norm_stats['node_features']['input_slew']['mean']
                        ) / norm_stats['node_features']['input_slew']['std']
                    
                    # Normalize output_load (column 6)
                    load_mask = normalized[:, 6] != 0
                    if load_mask.any():
                        normalized[load_mask, 6] = (
                            normalized[load_mask, 6] - norm_stats['node_features']['output_load']['mean']
                        ) / norm_stats['node_features']['output_load']['std']
                # Apply adjacency matrix
                aggregated_features = torch.matmul(adjacency_matrix, normalized)
                
                data = Data(
                    x=aggregated_features,
                    edge_index=edge_index
                )
                return data
            
            center_graph = create_center_graph(support_graphs[0])
            center_data = create_pyg_data_simple(center_graph, saved_norm_stats)
            center = gnn_model(Batch.from_data_list([center_data]).to(device)).item()
            y_max = max(y_norm)
            y_min = min(y_norm)
            #print(f"center:{center}")
            # Get model predictions for scaling
            all_preds = []
            for graph in all_task_graphs:
                data = create_pyg_data_simple(graph, saved_norm_stats)
                with torch.no_grad():
                    pred = gnn_model(Batch.from_data_list([data]).to(device)).item()
                    all_preds.append(pred)
            
            min_val = min(all_preds)
            max_val = max(all_preds)
            #print(min_val,max_val)
            if abs(max_val - min_val) > 1e-8:
                grad = (y_max - y_min) / (max_val - min_val)
                
                # Calculate move parameter
                move = center - y_norm[2] / grad
                
                # Run evaluation with same structure as reference notebook
                (total_loss1, inter_loss1, leftex_loss1, rightex_loss1,
                 mape_loss1, leftex_mape1, inter_mape1, rightex_mape1,
                 predictions, actual_values, _, _, _, _) = plot_sampled_performance_gnn(
                    gnn_model, support_graphs, y_norm,
                    all_task_graphs, all_task_outputs,
                    grad, move,y_mean,y_std, norm_stats=saved_norm_stats
                )
                #print(predictions)
                # Add to global collections
                all_predictions_global.extend(predictions)
                all_actuals_global.extend(actual_values)
                # print(f"pre:{predictions}")
                # print(f"act:{actual_values}")
                # Calculate NRMSE values
                nrmse1 = (total_loss1 ** 0.5) / (abs(y1_mean) + 1e-4) * 100
                nrmse_inter = (inter_loss1 ** 0.5) / (abs(y_inter_mean) + 1e-4) * 100
                nrmse_leftex = (leftex_loss1 ** 0.5) / (abs(y_leftex_mean) + 1e-4) * 100
                nrmse_rightex = (rightex_loss1 ** 0.5) / (abs(y_rightex_mean) + 1e-4) * 100
                mape_percent = mape_loss1 * 100
                
                if nrmse1 > 20:
                    high_error.append(task_id)
                
                total_nrmse.append(nrmse1.item())
                total_extra_l.append(nrmse_leftex.item())
                total_extra_r.append(nrmse_rightex.item())
                total_inter.append(nrmse_inter.item())
                total_mape.append(mape_percent)
                total_mape_l.append(leftex_mape1 * 100)
                total_mape_r.append(rightex_mape1 * 100)
                total_mape_inter.append(inter_mape1 * 100)
                print(nrmse1.item())
                if i % 20 == 0 and i > 0:
                    print(f"  Current avg NRMSE: {sum(total_nrmse)/len(total_nrmse):.2f}%, Tasks completed: {len(total_nrmse)}")
    
    except Exception as e:
        print(f"Error processing task {task_id}: {e}")
        continue

print(f"✅ Completed processing {len(total_nrmse)} valid tasks")
print(f"📈 Total global predictions collected: {len(all_predictions_global)}")

🔄 Processing 100 continuous tasks...
Processing task 1/100 (index: 54)
1.4284789562225342
0.8505473732948303
0.4405736029148102
2.1256041526794434
2.6544883251190186
0.8554076552391052
1.9496463537216187
0.711129903793335
0.7344969511032104
1.1620912551879883
Processing task 11/100 (index: 985)
1.297892451286316
0.6468129754066467
1.2718174457550049
0.8493901491165161
0.9354029893875122
0.4722850024700165
0.9499261379241943
1.5544317960739136
0.6170907020568848
0.5815362930297852
Processing task 21/100 (index: 526)
0.3922449052333832
  Current avg NRMSE: 1.56%, Tasks completed: 46
0.9366656541824341
0.9273408651351929
1.2144831418991089
1.264650583267212
0.5739200115203857
1.2603610754013062
0.9362914562225342
0.9501955509185791
1.553780436515808
Processing task 31/100 (index: 843)
0.9606773853302002
1.9033212661743164
1.159837245941162
0.7554702162742615
1.511013388633728
2.0397157669067383
2.059596538543701
1.1189024448394775
0.73201584815979
0.9363148212432861
Processing task 41/100

In [ ]:
# Print final results (same format as reference notebook)
print(f"\n=== Final Results on {len(total_nrmse)} valid tasks =====")
if len(total_nrmse) > 0:
    print(f"Average Total NRMSE: {sum(total_nrmse)/len(total_nrmse):.3f}%")
    print(f"Average Left Extrapolation NRMSE: {sum(total_extra_l)/len(total_extra_l):.3f}%")
    print(f"Average Right Extrapolation NRMSE: {sum(total_extra_r)/len(total_extra_r):.3f}%")
    print(f"Average Interpolation NRMSE: {sum(total_inter)/len(total_inter):.3f}%")
    print(f"Average MAPE: {sum(total_mape)/len(total_mape):.3f}%")
    print(f"Average Left MAPE: {sum(total_mape_l)/len(total_mape_l):.3f}%")
    print(f"Average Inter MAPE: {sum(total_mape_inter)/len(total_mape_inter):.3f}%")
    print(f"Average Right MAPE: {sum(total_mape_r)/len(total_mape_r):.3f}%")
else:
    print("No valid tasks processed")


=== Final Results on 0 valid tasks =====
No valid tasks processed


In [ ]:
# Create final combined prediction vs actual scatter plot with global R²
if len(all_predictions_global) > 0 and len(all_actuals_global) > 0:
    plt.figure(figsize=(10, 8))
    plt.scatter(all_actuals_global, all_predictions_global, alpha=0.6, s=30)

    # Plot y=x line for perfect prediction reference
    min_val = min(min(all_actuals_global), min(all_predictions_global))
    max_val = max(max(all_actuals_global), max(all_predictions_global))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction (y=x)')

    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title(f'GNN MAML Model ({process_type}_{corner_type}): Global Prediction vs Actual Values')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Make the plot square
    plt.axis('equal')

    # Calculate R² score for all combined data
    actual_np = np.array(all_actuals_global)
    pred_np = np.array(all_predictions_global)
    
    # Filter out any NaN or infinite values
    valid_mask = ~(np.isnan(actual_np) | np.isnan(pred_np) | np.isinf(actual_np) | np.isinf(pred_np))
    actual_clean = actual_np[valid_mask]
    pred_clean = pred_np[valid_mask]
    
    if len(actual_clean) > 0 and np.var(actual_clean) > 0:
        r2 = 1 - np.sum((actual_clean - pred_clean)**2) / np.sum((actual_clean - np.mean(actual_clean))**2)
        r2_text = f'R² = {r2:.4f}'
    else:
        r2_text = 'R² = N/A'
        r2 = float('nan')
    
    plt.text(0.1, 0.9, f'{r2_text}\nTotal Points: {len(actual_clean)}\nTasks: {len(total_nrmse)}', 
             transform=plt.gca().transAxes, 
             fontsize=12,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()
    
    print(f"\n🎯 Global R² Score: {r2:.4f}" if not np.isnan(r2) else "\n⚠️ Could not calculate R² score")
    print(f"📊 Total data points: {len(actual_clean)}")
    print(f"🔢 Total valid tasks processed: {len(total_nrmse)}")
else:
    print("⚠️ No global predictions collected for R² calculation")

⚠️ No global predictions collected for R² calculation


In [ ]:
# Save results to CSV files (optional)
results_dir = f'gnn_maml_validation_results_{process_type}_{corner_type}'
os.makedirs(results_dir, exist_ok=True)

if len(total_nrmse) > 0:
    with open(f'{results_dir}/test_total_NRMSE.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(total_nrmse)
    
    with open(f'{results_dir}/test_inter_NRMSE.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(total_inter)
    
    with open(f'{results_dir}/test_extra_l_NRMSE.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(total_extra_l)
    
    with open(f'{results_dir}/test_extra_r_NRMSE.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(total_extra_r)
    
    with open(f'{results_dir}/test_MAPE.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(total_mape)
    
    with open(f'{results_dir}/test_high_error.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(high_error)
    
    print(f"📁 Results saved to {results_dir}/")
else:
    print("⚠️ No results to save")

⚠️ No results to save


In [ ]:
# Summary statistics
print("\n📈 Summary Statistics:")
if len(total_nrmse) > 0:
    results_summary = {
        'Model Type': f'GNN MAML ({process_type}_{corner_type})',
        'Total Tasks Processed': len(total_nrmse),
        'Global Data Points': len(all_predictions_global),
        'Average Total NRMSE (%)': sum(total_nrmse)/len(total_nrmse),
        'Average Interpolation NRMSE (%)': sum(total_inter)/len(total_inter),
        'Average Left Extrapolation NRMSE (%)': sum(total_extra_l)/len(total_extra_l),
        'Average Right Extrapolation NRMSE (%)': sum(total_extra_r)/len(total_extra_r),
        'Average MAPE (%)': sum(total_mape)/len(total_mape),
        'Average Left MAPE (%)': sum(total_mape_l)/len(total_mape_l),
        'Average Inter MAPE (%)': sum(total_mape_inter)/len(total_mape_inter),
        'Average Right MAPE (%)': sum(total_mape_r)/len(total_mape_r),
        'Global R² Score': r2 if 'r2' in locals() and not np.isnan(r2) else 'N/A',
        'High Error Tasks': len(high_error)
    }
    
    for key, value in results_summary.items():
        if isinstance(value, float) and key != 'Global R² Score':
            print(f"{key}: {value:.3f}")
        else:
            print(f"{key}: {value}")
else:
    print("No valid results to summarize")


📈 Summary Statistics:
No valid results to summarize


In [ ]:
# Save numpy arrays for later analysis
if len(all_predictions_global) > 0 and len(all_actuals_global) > 0:
    np.save(f"{results_dir}/GNN_{process_type}_{corner_type}_pred.npy", all_predictions_global)
    np.save(f"{results_dir}/GNN_{process_type}_{corner_type}_act.npy", all_actuals_global)
    print(f"\n💾 Saved prediction and actual arrays to {results_dir}/")